# Train Earbud Detect COCO — Kaggle Run All

Notebook này dành cho dataset **Roboflow COCO** có cấu trúc `train/valid/test` và file `_annotations.coco.json`.

Bạn chỉ cần làm 4 việc:

1. Add Input dataset trên Kaggle.
2. Bật **Settings → Accelerator → GPU** và bật Internet để tải pretrained weight.
3. Sửa duy nhất danh sách `DATASET_ROOTS` trong **CELL 1**. Có thể khai báo một hoặc nhiều dataset COCO.
4. Chọn **Run All**.

Notebook tự chuyển COCO sang YOLO, train, đánh giá và tạo `best_earbud_detector.pt` cùng `earbud_training_results.zip` trong `/kaggle/working`.

In [ ]:
# ================================================================
# CELL 1 — CHỈ SỬA DATASET_ROOTS, SAU ĐÓ BẤM RUN ALL
# ================================================================
DATASET_ROOTS = [
    r'/kaggle/input/CHANGE_ME/earbud-detect.coco',
    # r'/kaggle/input/CHANGE_ME/left-right-earbud.coco',
]

# Các giá trị dưới đây đã được cấu hình sẵn cho Kaggle GPU.
MODEL_WEIGHTS = 'yolo11s.pt'
EPOCHS = 80
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 20
WORKERS = 2
SEED = 42
RUN_NAME = 'earbud_coco_detector'
STRICT_GEOMETRY_V2 = False  # Đổi True chỉ khi đã gán đúng 6 nhãn geometry v2

In [ ]:
# CELL 2 — Cài thư viện
%pip install -q -U ultralytics

In [ ]:
# CELL 3 — Kiểm tra đường dẫn và GPU
import hashlib
import json
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import torch

DATASET_ROOTS = [Path(path) for path in DATASET_ROOTS]
assert DATASET_ROOTS, 'DATASET_ROOTS phải có ít nhất một đường dẫn.'
for dataset_root in DATASET_ROOTS:
    assert dataset_root.is_dir(), (
        f'Không tìm thấy dataset: {dataset_root}\n'
        'Mở panel Input của Kaggle, copy đường dẫn thư mục chứa train/valid/test '
        'và dán vào DATASET_ROOTS ở CELL 1.'
    )
assert torch.cuda.is_available(), (
    'GPU chưa được bật. Vào Settings → Accelerator → chọn GPU rồi Run All lại.'
)
print('Datasets:')
for dataset_root in DATASET_ROOTS:
    print('  -', dataset_root)
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)

In [ ]:
# CELL 4 — Kiểm tra và chuyển Roboflow COCO sang YOLO
ANNOTATION_FILE = '_annotations.coco.json'
SPLIT_ALIASES = {
    'train': ('train', 'training'),
    'val': ('valid', 'val', 'validation'),
    'test': ('test', 'testing'),
}
GEOMETRY_V2_ORDER = [
    'open_case', 'close_case', 'left_earbud', 'right_earbud',
    'empty_left', 'empty_right'
]
LEGACY_ORDER = [
    'Earphone_Case', 'Earbud', 'Empty_Slot',
    'Left_Earbud', 'Right_Earbud', 'Hand'
]
GEOMETRY_V2_CLASSES = set(GEOMETRY_V2_ORDER)
LEGACY_CLASSES = set(LEGACY_ORDER)

dataset_sources = []
category_counts = Counter()
train_counts = Counter()
all_splits = set()
all_unused_categories = set()
for source_index, dataset_root in enumerate(DATASET_ROOTS):
    children = {p.name.lower(): p for p in dataset_root.iterdir() if p.is_dir()}
    split_dirs = {}
    for canonical, aliases in SPLIT_ALIASES.items():
        for alias in aliases:
            candidate = children.get(alias.lower())
            if candidate is not None and (candidate / ANNOTATION_FILE).is_file():
                split_dirs[canonical] = candidate
                break
    assert 'train' in split_dirs and 'val' in split_dirs, (
        f'{dataset_root}: cần train và valid/val, mỗi split phải có '
        f'{ANNOTATION_FILE}. Đã tìm thấy: {list(split_dirs)}'
    )

    payloads = {}
    category_names = {}
    used_category_ids = set()
    for split, split_dir in split_dirs.items():
        with (split_dir / ANNOTATION_FILE).open(encoding='utf-8') as file:
            payload = json.load(file)
        payloads[split] = payload
        for category in payload.get('categories', []):
            category_id = int(category['id'])
            category_name = str(category['name']).strip()
            previous = category_names.get(category_id)
            assert previous in (None, category_name), (
                f'{dataset_root}: category_id={category_id} đổi tên giữa các split: '
                f'{previous!r} != {category_name!r}'
            )
            category_names[category_id] = category_name
        split_counts_by_id = Counter(
            int(annotation['category_id'])
            for annotation in payload.get('annotations', [])
        )
        unknown_ids = sorted(set(split_counts_by_id) - set(category_names))
        assert not unknown_ids, (
            f'{dataset_root}/{split}: category_id không tồn tại: {unknown_ids}'
        )
        used_category_ids.update(split_counts_by_id)
        split_counts_by_name = Counter({
            category_names[category_id]: count
            for category_id, count in split_counts_by_id.items()
        })
        category_counts.update(split_counts_by_name)
        if split == 'train':
            train_counts.update(split_counts_by_name)
    all_unused_categories.update(
        name for category_id, name in category_names.items()
        if category_id not in used_category_ids
    )
    all_splits.update(split_dirs)
    dataset_sources.append({
        'index': source_index,
        'root': dataset_root,
        'split_dirs': split_dirs,
        'payloads': payloads,
        'category_names': category_names,
    })

assert category_counts, 'Các dataset không có bounding box.'
all_class_names = set(category_counts)
if all_class_names <= LEGACY_CLASSES:
    class_names = [name for name in LEGACY_ORDER if name in all_class_names]
elif all_class_names <= GEOMETRY_V2_CLASSES:
    class_names = [name for name in GEOMETRY_V2_ORDER if name in all_class_names]
else:
    raise AssertionError(
        'Không được trộn schema legacy với geometry v2 hoặc tên lớp lạ. '
        f'Các lớp nhận được: {sorted(all_class_names)}'
    )
missing_train = [name for name in class_names if train_counts[name] == 0]
assert not missing_train, f'Các class không có box trong train: {missing_train}'
class_to_id = {name: index for index, name in enumerate(class_names)}
if all_unused_categories:
    print('Bỏ category metadata không có box:', sorted(all_unused_categories))

is_geometry_v2 = set(class_names) == GEOMETRY_V2_CLASSES
if STRICT_GEOMETRY_V2:
    assert is_geometry_v2, (
        f'Dataset chưa đúng geometry v2. Hiện có: {class_names}; '
        f'cần: {sorted(GEOMETRY_V2_CLASSES)}'
    )
elif not is_geometry_v2:
    print('WARNING: Dataset train được detector baseline nhưng chưa phải geometry v2.')
    print('Hiện có:', class_names)
    print('Geometry v2 cần:', sorted(GEOMETRY_V2_CLASSES))

OUTPUT_ROOT = Path('/kaggle/working/earbud_yolo_prepared')
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True)
report = {
    'sources': [str(source['root']) for source in dataset_sources],
    'class_names': class_names,
    'boxes_per_class_before_dedup': dict(category_counts),
    'splits': {},
    'skipped_invalid_boxes': 0,
    'skipped_duplicate_images': 0,
}
seen_image_hashes = {}
for source in dataset_sources:
    source_index = source['index']
    prefix = f's{source_index:02d}_'
    for split, split_dir in source['split_dirs'].items():
        payload = source['payloads'][split]
        category_names = source['category_names']
        images = {int(row['id']): row for row in payload.get('images', [])}
        annotations = defaultdict(list)
        for annotation in payload.get('annotations', []):
            image_id = int(annotation['image_id'])
            assert image_id in images, f'{split}: image_id={image_id} không tồn tại'
            annotations[image_id].append(annotation)

        images_out = OUTPUT_ROOT / split / 'images'
        labels_out = OUTPUT_ROOT / split / 'labels'
        images_out.mkdir(parents=True, exist_ok=True)
        labels_out.mkdir(parents=True, exist_ok=True)
        split_report = report['splits'].setdefault(split, {'images': 0, 'boxes': 0})

        for image_id, image in images.items():
            safe_name = Path(str(image['file_name']).replace('\\', '/')).name
            candidates = (split_dir / safe_name, split_dir / 'images' / safe_name)
            source_image = next((path for path in candidates if path.is_file()), None)
            assert source_image is not None, f'Không tìm thấy ảnh {safe_name} trong {split_dir}'
            image_hash = hashlib.sha256(source_image.read_bytes()).hexdigest()
            if image_hash in seen_image_hashes:
                report['skipped_duplicate_images'] += 1
                continue
            seen_image_hashes[image_hash] = f'{source_index}/{split}/{safe_name}'
            output_stem = prefix + source_image.stem
            output_image_name = output_stem + source_image.suffix.lower()
            label_name = output_stem + '.txt'
            shutil.copy2(source_image, images_out / output_image_name)

            width = float(image.get('width', 0))
            height = float(image.get('height', 0))
            assert width > 0 and height > 0, f'Kích thước ảnh không hợp lệ: {safe_name}'
            label_lines = []
            for annotation in annotations.get(image_id, []):
                bbox = annotation.get('bbox', [])
                if len(bbox) != 4:
                    report['skipped_invalid_boxes'] += 1
                    continue
                x, y, box_width, box_height = map(float, bbox)
                x1, y1 = max(0.0, min(width, x)), max(0.0, min(height, y))
                x2 = max(0.0, min(width, x + box_width))
                y2 = max(0.0, min(height, y + box_height))
                if x2 <= x1 or y2 <= y1:
                    report['skipped_invalid_boxes'] += 1
                    continue
                values = (
                    ((x1 + x2) / 2.0) / width,
                    ((y1 + y2) / 2.0) / height,
                    (x2 - x1) / width,
                    (y2 - y1) / height,
                )
                category_id = int(annotation['category_id'])
                class_name = category_names[category_id]
                class_id = class_to_id[class_name]
                label_lines.append(
                    f'{class_id} ' + ' '.join(f'{value:.8f}' for value in values)
                )
                split_report['boxes'] += 1
            (labels_out / label_name).write_text(
                '\n'.join(label_lines) + ('\n' if label_lines else ''),
                encoding='utf-8',
            )
            split_report['images'] += 1

yaml_lines = [
    f'path: {OUTPUT_ROOT.as_posix()}',
    'train: train/images',
    'val: val/images',
]
if 'test' in all_splits:
    yaml_lines.append('test: test/images')
yaml_lines.extend(('', f'nc: {len(class_names)}', 'names:'))
yaml_lines.extend(
    f'  {index}: {json.dumps(name, ensure_ascii=False)}'
    for index, name in enumerate(class_names)
)
YAML_PATH = OUTPUT_ROOT / 'data.yaml'
YAML_PATH.write_text('\n'.join(yaml_lines) + '\n', encoding='utf-8')
(OUTPUT_ROOT / 'dataset_report.json').write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)

print('\n========== DATASET REPORT ==========')
for index, name in enumerate(class_names):
    box_count = report['boxes_per_class_before_dedup'][name]
    print(f'{index}: {name} — {box_count} boxes')
for split, counts in report['splits'].items():
    image_count, box_count = counts['images'], counts['boxes']
    print(f'{split}: {image_count} images, {box_count} boxes')
print('Invalid boxes skipped:', report['skipped_invalid_boxes'])
print('Duplicate images skipped:', report['skipped_duplicate_images'])
print('YOLO data.yaml:', YAML_PATH)
print('====================================')

In [ ]:
# CELL 5 — Train YOLO trên GPU
from ultralytics import YOLO

local_weights = [
    path for dataset_root in DATASET_ROOTS
    for path in dataset_root.rglob(MODEL_WEIGHTS)
]
model_source = str(local_weights[0]) if local_weights else MODEL_WEIGHTS
print('Pretrained weights:', model_source)
TRAINING_ROOT = Path('/kaggle/working/training')
RUN_DIR = TRAINING_ROOT / RUN_NAME
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
model = YOLO(model_source)
train_results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    workers=WORKERS,
    project=str(TRAINING_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    cache='disk',
    fliplr=0.0,  # Không lật ảnh làm sai ngữ nghĩa left/right
    flipud=0.0,
    amp=True,
    plots=True,
    save=True,
    save_period=10,
    verbose=True,
)

In [ ]:
# CELL 6 — Đánh giá best.pt
RUN_DIR = Path('/kaggle/working/training') / RUN_NAME
BEST_PATH = RUN_DIR / 'weights' / 'best.pt'
assert BEST_PATH.is_file(), f'Không tìm thấy checkpoint: {BEST_PATH}'
evaluation_split = 'test' if 'test' in all_splits else 'val'
best_model = YOLO(str(BEST_PATH))
metrics = best_model.val(
    data=str(YAML_PATH),
    split=evaluation_split,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    plots=True,
    project='/kaggle/working/training',
    name=f'{RUN_NAME}_{evaluation_split}',
)
print('\n========== TEST METRICS ==========')
print('Split:', evaluation_split)
print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')
print('==================================')

In [ ]:
# CELL 7 — Đóng gói để tải về
PORTABLE_BEST = Path('/kaggle/working/best_earbud_detector.pt')
shutil.copy2(BEST_PATH, PORTABLE_BEST)
shutil.copy2(YAML_PATH, RUN_DIR / 'data_used.yaml')
shutil.copy2(OUTPUT_ROOT / 'dataset_report.json', RUN_DIR / 'dataset_report.json')
summary = {
    'model': MODEL_WEIGHTS,
    'classes': class_names,
    'evaluation_split': evaluation_split,
    'map50': float(metrics.box.map50),
    'map50_95': float(metrics.box.map),
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
}
(RUN_DIR / 'training_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
archive = shutil.make_archive(
    '/kaggle/working/earbud_training_results', 'zip', root_dir=RUN_DIR
)
print('\nTRAIN HOÀN TẤT')
print('Tải hai file trong mục Output:')
print('  ', PORTABLE_BEST)
print('  ', archive)

In [ ]:
# CELL 8 — Hiển thị biểu đồ kết quả
from IPython.display import Image, display

for file_name in ('results.png', 'confusion_matrix_normalized.png', 'PR_curve.png'):
    image_path = RUN_DIR / file_name
    if image_path.is_file():
        print(file_name)
        display(Image(filename=str(image_path)))